# SmolVLA post-training: V0 / V1 / V2
Runtime → Change runtime type → **GPU** (T4 works; L4/A100 are faster). Datasets and checkpoints live on Google Drive under `DRIVE_DIR`, so a disconnect only loses the steps since the last checkpoint: re-run all cells and training resumes.

Before the first run, upload the exported datasets (`data/lerobot/sim_seen`, `data/lerobot/phone`) to `DRIVE_DIR/data/lerobot/`.

In [ ]:
# ---- configuration ----
CONFIG = "V0"            # V0: sim only | V1: + phone (naive) | V2: + phone (ambient)
STEPS = 3000
BATCH, GRAD_ACCUM = 32, 1   # T4 (16 GB): try 32 x 1; drop to 16 x 2 if out of memory
AMBIENT_T_MIN = 0.5         # used by V2 only
EVAL_K = 20                 # episodes per task (20 intermediate, 50 final)
REPO_URL = "https://github.com/<user>/demo_conditioned_ambient_diffusion.git"  # set to the public repo
DRIVE_DIR = "/content/drive/MyDrive/dcad"
DATA = {"V0": ["sim_seen"], "V1": ["sim_seen", "phone"], "V2": ["sim_seen", "phone"]}[CONFIG]
T_MIN = AMBIENT_T_MIN if CONFIG == "V2" else 0.0

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
if not os.path.exists("/content/repo"):
    !git clone -q $REPO_URL /content/repo
%cd /content/repo
!git pull -q
!pip install -q -r requirements.txt
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
# copy datasets to local disk: decoding video from the Drive mount is slow
for name in DATA:
    src, dst = f"{DRIVE_DIR}/data/lerobot/{name}", f"/content/repo/data/lerobot/{name}"
    if not os.path.exists(dst):
        !mkdir -p /content/repo/data/lerobot && cp -r "$src" "$dst"
DATA_ARGS = " ".join(f"data/lerobot/{n}" for n in DATA)
OUT = f"{DRIVE_DIR}/checkpoints/vla"
print(DATA_ARGS, "->", OUT)

## Timing check (run once per GPU type)
100 steps into a scratch run; it reports seconds/step. 500 steps ≈ 5 × this cell's time.

In [ ]:
!python vla/train_smolvla.py --run_name timing_$CONFIG --data $DATA_ARGS --out /content/timing --steps 100 \
    --batch $BATCH --grad_accum $GRAD_ACCUM --warmup 10 --log_every 10 --save_every 100000 --device cuda
!tail -3 /content/timing/timing_$CONFIG/loss.csv

## Train (resumes automatically from the latest checkpoint on Drive)

In [ ]:
!python vla/train_smolvla.py --run_name $CONFIG --data $DATA_ARGS --out $OUT --steps $STEPS \
    --batch $BATCH --grad_accum $GRAD_ACCUM --ambient_t_min $T_MIN --save_every 250 --resume --device cuda

## Evaluate checkpoints (seen vs held-out blue tasks)

In [ ]:
import glob, re
steps = sorted(int(re.search(r"step_(\d+)", d).group(1)) for d in glob.glob(f"{OUT}/{CONFIG}/step_*"))
print(steps)
for s in steps[1::2] + ([steps[-1]] if len(steps) % 2 else []):  # every other checkpoint and the last
    !python vla/eval_smolvla.py --run $OUT/$CONFIG --step $s --k $EVAL_K --out $DRIVE_DIR/outputs/vla/$CONFIG/step_$s --device cuda